<a href="https://colab.research.google.com/github/watanabe3tipapa/GoogleColaboratory/blob/main/mcp/colab_mcp_marimo_pwa_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab MCP Server × marimo × PWA 連携アーキテクチャ探査Google Colab上で動作するMCPサーバ（marimo）と、専用PWAクライアントアプリとの間で連携作業を実現するための手法調査と推奨構成の提案📅 2026年9月8日 📝 技術調査レポート 🔖 MCP / marimo / PWA

## 前提と制約の整理### 1.1 各コンポーネントの役割| コンポーネント | 本来の役割 | 注意点 || --- | --- | --- || colab-mcp (googlecolab/colab-mcp) | ローカルマシンで動くMCPサーバ。ローカルエージェント（Claude Code, Gemini CLIなど）と、ブラウザ内のColabセッションを橋渡しする。 | PWA（ブラウザ）から直接使う設計ではない。ローカル→Colabの方向性が前提。 || marimo | Pythonのリアクティブノートブック。`--mcp` フラグ付きで起動すると、MCPサーバとして機能し、`/mcp/server` エンドポイントを提供する。 | Colab上で `marimo edit --mcp --headless` で起動可能。 || PWA | プログレッシブウェブアプリ。ブラウザ上で動作し、オフライン対応やネイティブライクなUIが可能。 | MCPクライアントとして振る舞う必要がある。 |### 1.2 技術的制約- 🔒 **Colabのネットワーク**: 外部から直接アクセスできない。`google.colab.kernel.proxyPort()` でプロキシURLを発行する必要がある。- 🌐 **CORS**: ブラウザから別オリジンのMCPサーバにアクセスする場合、CORS対応が必須。- 📡 **MCPトランスポート**: ブラウザからは HTTP/SSE または WebSocket が現実的。Stdioは使えない。- ⏱️ **セッション継続性**: Colabは一定時間アイドルで切断される。長時間のMCP接続には工夫が必要。

## 提案アーキテクチャ（4パターン）### パターンA: PWA → Colab上のmarimo MCPサーバ（直接HTTP接続）最もシンプルな構成。PWAがブラウザ内のMCPクライアントとして、Colab上のmarimoのMCPエンドポイントに直接接続する。```PWA (MCP Client) ←──HTTP/SSE──→ Colab上のmarimo (MCP Server)```

#### Colab側（サーバ）

In [ ]:
# Colab セルで実行!pip -q install -U "marimo[mcp]"import secretsfrom google.colab.output import eval_jsPORT = 8888TOKEN = secrets.token_urlsafe(16)NOTEBOOK = "/content/notebook.py"# バックグラウンドでmarimo MCPサーバを起動!marimo edit {NOTEBOOK}   --headless   --host 0.0.0.0   --port {PORT}   --token-password "{TOKEN}"   --mcp   --mcp-allow-remote   > /content/marimo.log 2>&1 &# プロキシURL取得proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")print(f"MCP Endpoint: {proxy_url}/mcp/server?access_token={TOKEN}")

#### PWA側（クライアント / React例）```typescriptimport { useMcp } from 'use-mcp/react';function App() {  const { state, tools, callTool } = useMcp({    url: 'https://xxxx-colab-googleusercontent.com/mcp/server?access_token=TOKEN'  });  return (    <div>      {state === 'ready' && (        <button onClick={() => callTool('active_notebooks', {})}>          ノートブック一覧取得        </button>      )}    </div>  );}```| 観点 | 評価 || --- | --- || メリット | 構成がシンプル。追加インフラ不要。marimoのAIツール（ノートブック読み取り、エラー集計など）をPWAから直接利用可能。 || デメリット | CORS対応が必要。ColabのプロキシURLはセッションごとに変わるためPWAに都度入力が必要。トークン埋め込みはセュリティリスク。 |

### パターンB: 公開トンネル経由で接続（ngrok / Cloudflare Tunnel）Colab上のmarimo MCPサーバを、外部からアクセス可能なURLに公開し、PWAからそこに接続する。```PWA (MCP Client) ←──HTTPS──→ ngrok / Cloudflare Tunnel ←──→ Colab上のmarimo```

In [ ]:
# Colab セルで実行!pip -q install -U "marimo[mcp]" pyngrokfrom pyngrok import ngrokimport secretsPORT = 8888TOKEN = secrets.token_urlsafe(16)# ngrokでトンネル作成public_url = ngrok.connect(PORT, "http").public_urlprint(f"Public URL: {public_url}/mcp/server?access_token={TOKEN}")# marimo起動!marimo edit /content/notebook.py   --headless --host 0.0.0.0 --port {PORT}   --token-password "{TOKEN}" --mcp --mcp-allow-remote

| 観点 | 評価 || --- | --- || メリット | URLが固定（トンネル存続中）で、PWAに組み込みやすい。パターンAより接続が安定。 || デメリット | インターネットに公開されるためトークン認証・IP制限が必須。ngrok無料版は接続数・時間に制限あり。Colabのセッション制限に抵触する可能性。 |

### パターンC: PWAがMCPサーバ、ColabがMCPクライアント（逆方向）`@teatak/mcp-server-browser` のように、PWA側がMCPサーバとなり、Colab側（Python）から接続して作業を指示する構成。```Colab (Python MCP Client) ←──WebSocket──→ PWA (MCP Server in Browser)```#### PWA側（MCPサーバ）```typescriptimport { createServer } from "@teatak/mcp-server-browser";const server = createServer({  endpoint: "wss://your-bridge-server.com/mcp/ws",  serverInfo: { name: "pwa-client", version: "1.0.0" },});server.registerTool({  name: "get_user_input",  description: "PWAからユーザーの入力を取得",  inputSchema: { type: "object", properties: {} },  handler: async () => {    const result = await showPromptDialog(); // PWAのUI関数    return { content: [{ type: "text", text: result }] };  },});server.connect();```| 観点 | 評価 || --- | --- || メリット | PWAの能力（カメラ、GPS、通知、IndexedDBなど）をColab/Python側から利用可能。人間の確認を挟みやすい。 || デメリット | ブリッジサーバ（WebSocket中継）が別途必要。データの流れが逆になるため、ノートブック操作をPWAから行う用途には不向き。 |

### パターンD: colab-mcpの本来の使い方（ローカルエージェント経由）colab-mcpは「ローカルエージェント→Colab」方向の設計。PWAを「ローカルエージェントの代替」として使うには橋渡しが必要。```PWA ←──Web/API──→ ローカルMCPブリッジ (Node/Python) ←──Stdio──→ colab-mcp ←──→ Colabセッション```| 観点 | 評価 || --- | --- || メリット | Google公式のcolab-mcpをそのまま活用。ローカルマシンのリソースを使わずにColab上でPythonコードを実行可能。 || デメリット | ブラウザからローカルホストへの接続はCORSやセキュリティ制約が厳しい。PWAよりもElectronやTauri製のデスクトップアプリで実装する方が現実的。 |

## 推奨構成と実装ステップ✅ **推奨**: パターンA + パターンBのハイブリッドが最も現実的です。開発時はColabプロキシURLをPWAに手動入力し、本番運用時にはCloudflare Tunnel等で固定URL化することで、PWAからのシームレスな連携が実現できます。

### ステップ1: Colab上でmarimo MCPサーバを起動

In [ ]:
# Colab セルで実行!pip -q install -U "marimo[mcp]"import secretsfrom google.colab.output import eval_jsPORT = 8888TOKEN = secrets.token_urlsafe(16)NOTEBOOK = "/content/app.py"!pkill -f "marimo" || true!marimo edit {NOTEBOOK}   --headless --host 0.0.0.0 --port {PORT}   --token-password "{TOKEN}" --mcp --mcp-allow-remote   > /content/marimo.log 2>&1 &proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")mcp_url = f"{proxy_url}/mcp/server?access_token={TOKEN}"print(f"\n=== MCP Server URL ===\n{mcp_url}\n")

### ステップ2: PWAプロジェクトの作成```bashnpx create-react-app marimo-pwa --template cra-template-pwacd marimo-pwanpm install use-mcp react-json-view```

### ステップ3: PWAでのMCP接続コンポーネント実装```tsximport React, { useState } from 'react';import { useMcp } from 'use-mcp/react';export function McpClient() {  const [serverUrl, setServerUrl] = useState('');  const { state, tools, callTool, error, log } = useMcp({    url: serverUrl,    debug: true,  });  return (    <div className="p-4">      <h1>marimo MCP PWA Client</h1>      <input        type="text"        value={serverUrl}        onChange={(e) => setServerUrl(e.target.value)}        placeholder="ColabのMCP URLを貼り付け"      />      <div>状態: {state}</div>      {state === 'ready' && tools?.map((tool) => (        <button key={tool.name}          onClick={() => callTool(tool.name, {})}>          実行: {tool.name}        </button>      ))}    </div>  );}```

### ステップ4: 証・セキュリティの強化URLクエリではなく `Sec-WebSocket-Protocol` ヘッダや `Authorization` ヘッダでトークンを送信します。PWAのService Workerでプロキシし、トークンを隠蔽することも検討してください。```typescriptconst { state, tools, callTool } = useMcp({  url: serverUrl.split('?')[0],  headers: {    'Authorization': `Bearer ${TOKEN}`  },  protocols: [`bearer.${TOKEN}`],});```

## 技術的課題と対策| 課題 | 対策 || --- | --- || CORSエラー | marimoを `--mcp-allow-remote` で起動。リバースプロキシ（nginx/Cloudflare）で `Access-Control-Allow-Origin` を設定。 || URLの短命性 | ngrok/Cloudflare Tunnelで固定ドメインを使用。または、QRコードでPWAにURLを簡単に引き渡す。 || トークン漏洩 | URLクエリではなく `Sec-WebSocket-Protocol` ヘッダや `Authorization` ヘッダで送信。PWAのService Workerでプロキシし、トークンを隠蔽。 || Colabセッション切断 | 定期的なping（WebSocketの場合）や、PWA側での再接続ロジックを実装。`use-mcp` は自動リトライに対応。 || MCPプロトコルの互換性 | marimoのMCPは実験的機能。PWA側も `use-mcp` など最新ライブラリを使用し、プロトコルバージョンを確認。 |

## 代替・関連手法### 📦 marimo-mcp (Vladisluv12/marimo-mcp)実行中のmarimoノートブックを自動検出し、読み取り・編集・実行ツールを公開するMCPサーバ。Colab上で動作させることで、より細かいノートブック操作が可能。### 🪄 marimo-magic既存のJupyter/Colab環境でmarimoセルを埋め込むことが可能。ただし、MCP連携にはパターンA/Bがより直接的。### ☁️ molab (marimo公式クラウド)Colabを使わず、marimo公式の無料クラウド環境を使う選択肢。marimoネイティブでMCPサポートが充実しており、PWA連携の実装が容易になる可能性がある。

## まとめ| 目的 | 推奨パターン || --- | --- || PWAからColab上のmarimoを直接操作したい | パターンA / B（marimo MCPサーバをHTTP/SSEで公開） || PWAの機能（カメラ、センサー）をColab/Pythonに使わせたい | パターンC（PWAがMCPサーバ） || Google公式のColab連携を最大限活用したい | パターンD（ただしPWAよりデスクトップアプリが適） |最も現実的なのは、Colab上で `marimo edit --mcp` を起動し、そのHTTPエンドポイントをPWAから `use-mcp` などのブラウザクライアントで接続するパターンA/B です。開発時はColabのプロキシURLを手動入力し、本番運用時にはCloudflare Tunnel等で固定URL化することで、PWAからのシームレスな連携が実現できます。本レポートは公開情報に基づく技術調査であり、構成の検証を前提としています。